In [ ]:
!pip install nltk scikit-learn pandas matplotlib seaborn

📂 Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

📊 Step 2: Load Dataset

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,Great acting,positive
1,Poor direction,negative
2,Average movie,neutral
3,Enjoyed every moment,positive
4,Absolutely awful,negative


🔍 Step 3: Data Understanding

In [3]:
print("Dataset Shape:", df.shape)

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

print("\nSample Data:")
print(df['review'][0])

Dataset Shape: (100, 2)

Class Distribution:
sentiment
positive    34
negative    33
neutral     33
Name: count, dtype: int64

Sample Data:
Great acting


🔄 Convert Labels to Numbers

In [4]:
df['sentiment'] = df['sentiment'].map({
    'negative': 0,
    'neutral': 1,
    'positive': 2
})

In [5]:
print(df['sentiment'].isnull().sum())

0


🧹 Step 4: NLP Preprocessing (VERY IMPORTANT)

In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove special characters & numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenization
    words = text.split()

    # Remove stopwords + Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

# Apply preprocessing
df['clean_text'] = df['review'].apply(preprocess_text)

df[['review','clean_text']].head()

,review,clean_text
0,Great acting,great acting
1,Poor direction,poor direction
2,Average movie,average movie
3,Enjoyed every moment,enjoyed every moment
4,Absolutely awful,absolutely awful


✂️ Step 5: Train-Test Split

In [7]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

🔢 Step 6: Feature Engineering
🟢 Bag of Words (BoW)

In [8]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

🔵 TF-IDF

In [9]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

🤖 Step 7: Model Training

1️⃣ Logistic Regression

In [10]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

2️⃣ Naive Bayes

In [11]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

3️⃣ Decision Tree

In [12]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

📈 Step 8: Evaluation Function

In [13]:
def evaluate_model(y_test, y_pred, model_name):

    print(f"\n📊 {model_name} Performance:")

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

    print("\nClassification Report:\n", classification_report(y_test, y_pred))

📊 Step 9: Evaluate All Models

In [14]:
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_dt, "Decision Tree")


📊 Logistic Regression Performance:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           1       1.00      1.00      1.00         5
           2       1.00      1.00      1.00         8

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


📊 Naive Bayes Performance:
Accuracy: 0.8
Precision: 0.8383333333333333
Recall: 0.8
F1 Score: 0.7929292929292929

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.57      0.73         7
           1       0.83      1.00      0.91         5
           2       0.70      0.88      0.78         8

    accuracy                           0.80        20
   macro avg       0.84      0.82      0.80        20
weighted avg       0.84      0.80      

🧠 Step 10: Comparison & Insights (WRITE THIS IN NOTEBOOK)


✔ Best Model

Logistic Regression usually performs best for text data

✔ Why TF-IDF works better than BoW

TF-IDF reduces importance of common words

Gives better semantic representation

✔ Preprocessing Impact

Removing stopwords improves performance
Lemmatization helps reduce word variations

✔ Trade-offs

**Logistic Regression**

Advantage: High accuracy

Disadvantage: Slightly slower


**Naive Bayes**

Advantage: Fast

Disadvantage: Assumes independence


**Decision Tree**

Advantage: Easy to understand

Disadvantage: Overfitting